In [5]:
# Initial setup follows last notebook, first_model_detailed,
# and skips where I messed up on which model to call
# and instead sets up the correct classification model from the start

In [29]:
import pandas as pd

In [30]:
from data.nbaDataImport import load_nba_games

In [31]:
nba_data = load_nba_games()

         id   game_date  season  home_team_id         home_team_name  \
0  18446819  2025-10-21    2025            21  Oklahoma City Thunder   
1  18446820  2025-10-21    2025            14     Los Angeles Lakers   
2  18446821  2025-10-22    2025            20        New York Knicks   
3  18446822  2025-10-22    2025             7       Dallas Mavericks   
4  18446823  2025-10-22    2025             4      Charlotte Hornets   

   visitor_team_id      visitor_team_name  home_team_score  \
0               11        Houston Rockets              125   
1               10  Golden State Warriors              109   
2                6    Cleveland Cavaliers              119   
3               27      San Antonio Spurs               92   
4                3          Brooklyn Nets              136   

   visitor_team_score  postseason  
0                 124       False  
1                 119       False  
2                 111       False  
3                 125       False  
4             

In [32]:
nba_data.head

<bound method NDFrame.head of             id   game_date  season  home_team_id         home_team_name  \
0     18446819  2025-10-21    2025            21  Oklahoma City Thunder   
1     18446820  2025-10-21    2025            14     Los Angeles Lakers   
2     18446821  2025-10-22    2025            20        New York Knicks   
3     18446822  2025-10-22    2025             7       Dallas Mavericks   
4     18446823  2025-10-22    2025             4      Charlotte Hornets   
...        ...         ...     ...           ...                    ...   
2638  18444560  2025-06-11    2024            12         Indiana Pacers   
2639  18444561  2025-06-13    2024            12         Indiana Pacers   
2640  18444562  2025-06-16    2024            21  Oklahoma City Thunder   
2641  18444563  2025-06-19    2024            12         Indiana Pacers   
2642  18444564  2025-06-22    2024            21  Oklahoma City Thunder   

      visitor_team_id      visitor_team_name  home_team_score  \
0   

In [33]:
nba_data.describe()

,id,season,home_team_id,visitor_team_id,home_team_score,visitor_team_score
count,2.643000e+03,2643.000000,2643.000000,2643.000000,2643.000000,2643.000000
mean,1.740806e+07,2024.500189,15.491487,15.479001,115.199773,113.377601
std,1.505938e+06,0.500095,8.570599,8.575434,12.840466,12.943088
min,1.590744e+07,2024.000000,1.000000,1.000000,77.000000,66.000000
25%,1.590810e+07,2024.000000,8.000000,8.000000,106.000000,105.000000
50%,1.844682e+07,2025.000000,16.000000,16.000000,115.000000,113.000000
75%,1.844748e+07,2025.000000,23.000000,23.000000,124.000000,122.000000
max,2.171614e+07,2025.000000,30.000000,30.000000,157.000000,162.000000


In [34]:
 # transform data for classifier model

In [35]:
# rename game_date column to date for readability
nba_data = nba_data.rename(columns={"game_date":"date"})

# One row per game from the home team's perspective.
home = nba_data[
    ["id", "date", "season", "home_team_id",
     "home_team_score", "visitor_team_score"]
].rename(columns={
    "home_team_id": "team_id",
    "home_team_score": "points_for",
    "visitor_team_score": "points_against"
})

# The same games from the visiting team's perspective.
away = nba_data[
    ["id", "date", "season", "visitor_team_id",
     "visitor_team_score", "home_team_score"]
].rename(columns={
    "visitor_team_id": "team_id",
    "visitor_team_score": "points_for",
    "home_team_score": "points_against"
})

# Stack the tables vertically, similar to SQL UNION ALL.
team_games = pd.concat([home, away], ignore_index=True)

team_games["date"] = pd.to_datetime(team_games["date"], utc=True)

team_games = team_games.sort_values(
    ["team_id", "season", "date", "id"]
).reset_index(drop=True)

In [36]:
# Create a win indicator

In [37]:
team_games["won"] = (
    team_games["points_for"] > team_games["points_against"]
).astype(int)

team_games["previous_game_won"] = (
    team_games
    .groupby(["team_id", "season"])["won"]
    .shift(1)
)

In [38]:
# more transformatoin: point differentials

In [39]:
team_games["date"] = pd.to_datetime(team_games["date"], utc=True)
team_games = team_games.sort_values(
    ["team_id", "season", "date", "id"]
).reset_index(drop=True)

team_games["point_differential"] = (
    team_games["points_for"] - team_games["points_against"]
)

groups = team_games.groupby(["team_id", "season"])

#win percentages BEFORE this game.
groups = team_games.groupby(["team_id", "season"])

team_games["season_win_percentage"] = groups["won"].transform(
    lambda s: s.shift(1).expanding().mean()
)

team_games["win_percentage_last_five_games"] = groups["won"].transform(
    lambda s: s.shift(1).rolling(5, min_periods=5).mean()
)

# Calendar days between game dates: back-to-back games = 1.
previous_date = groups["date"].shift(1)
team_games["rest_days_since_last_game"] = (
    team_games["date"].dt.normalize() -
    previous_date.dt.normalize()
).dt.days

# Average scoring margin over ALL earlier games this season.
team_games["average_point_differential"] = (
    groups["point_differential"].transform(
        lambda s: s.shift(1).expanding().mean()
    ))

In [40]:
#transformatoin part 3

for source, prefix in [
    ("points_for", "points_scored"),
    ("points_against", "points_allowed"),
]:
    team_games[f"{prefix}_per_game"] = groups[source].transform(
        lambda s: s.shift(1).expanding().mean()
    )
    team_games[f"{prefix}_last_five_games"] = groups[source].transform(
        lambda s: s.shift(1).rolling(5, min_periods=5).mean()
    )

In [41]:
#LAST STEP OF SETUP FROM LAST WEEK:
# Joining back to model table

features = [
    "season_win_percentage",
    "win_percentage_last_five_games",
    "rest_days_since_last_game",
    "average_point_differential",
    "points_scored_per_game",
    "points_scored_last_five_games",
    "points_allowed_per_game",
    "points_allowed_last_five_games"
]

# Select only identifiers and pregame features.
# Exclude current-game scores/results to avoid leakage.
team_features = team_games[["id", "team_id"] + features].copy()

home_features = team_features.rename(columns={
    "team_id": "home_team_id",
    **{column: f"home_{column}" for column in features},
})

visitor_features = team_features.rename(columns={
    "team_id": "visitor_team_id",
    **{column: f"visitor_{column}" for column in features},
})

nba_model_data = nba_data.merge(
    home_features,
    on=["id", "home_team_id"],
    how="left",
    validate="one_to_one",
).merge(
    visitor_features,
    on=["id", "visitor_team_id"],
    how="left",
    validate="one_to_one",
)

In [42]:
home_team = 0
nba_model_data["home_win"] = (nba_data["home_team_score"] > nba_data["visitor_team_score"]).astype(int)


In [43]:
# Now, add in some of the extra differential features I thought about last week:

In [44]:
nba_model_data["last5_win_pct_diff"] = (
    nba_model_data["home_win_percentage_last_five_games"]
    - nba_model_data["visitor_win_percentage_last_five_games"]
)

nba_model_data["season_point_margin_diff"] = (
    nba_model_data["home_average_point_differential"]
    - nba_model_data["visitor_average_point_differential"]
)

nba_model_data["last5_point_margin_diff"] = (
    (nba_model_data["home_points_scored_last_five_games"]
     - nba_model_data["home_points_allowed_last_five_games"])
    - (nba_model_data["visitor_points_scored_last_five_games"]
       - nba_model_data["visitor_points_allowed_last_five_games"])
)

nba_model_data["rest_days_diff"] = (
    nba_model_data["home_rest_days_since_last_game"]
    - nba_model_data["visitor_rest_days_since_last_game"]
)

In [45]:
# set features

In [46]:
nba_model_features = [
       'home_season_win_percentage',
       'home_win_percentage_last_five_games', 'home_rest_days_since_last_game',
       'home_average_point_differential', 'home_points_scored_per_game',
       'home_points_scored_last_five_games', 'home_points_allowed_per_game',
       'home_points_allowed_last_five_games', 'visitor_season_win_percentage',
       'visitor_win_percentage_last_five_games',
       'visitor_rest_days_since_last_game',
       'visitor_average_point_differential', 'visitor_points_scored_per_game',
       'visitor_points_scored_last_five_games',
       'visitor_points_allowed_per_game',
       'visitor_points_allowed_last_five_games',
       'last5_win_pct_diff',
       'season_point_margin_diff',
       'last5_point_margin_diff',
       'rest_days_diff'
] 

In [47]:
# drop NaNs

In [48]:
model_data = nba_model_data.dropna(subset=nba_model_features).copy()

In [50]:
model_data.head()

,id,date,season,home_team_id,home_team_name,visitor_team_id,visitor_team_name,home_team_score,visitor_team_score,postseason,...,visitor_average_point_differential,visitor_points_scored_per_game,visitor_points_scored_last_five_games,visitor_points_allowed_per_game,visitor_points_allowed_last_five_games,home_win,last5_win_pct_diff,season_point_margin_diff,last5_point_margin_diff,rest_days_diff
74,18446893,2025-10-31,2025,6,Cleveland Cavaliers,28,Toronto Raptors,101,112,False,...,-6.4,121.4,121.4,127.8,127.8,0,0.4,7.4,7.4,0.0
76,18446895,2025-10-31,2025,15,Memphis Grizzlies,14,Los Angeles Lakers,112,117,False,...,0.4,117.6,117.6,117.2,117.2,0,0.0,-3.0,-3.0,0.0
80,18446899,2025-11-01,2025,17,Milwaukee Bucks,26,Sacramento Kings,133,135,False,...,-5.8,111.0,111.0,116.8,116.8,0,0.6,12.6,12.6,-1.0
81,18446900,2025-11-01,2025,4,Charlotte Hornets,18,Minnesota Timberwolves,105,122,False,...,-4.8,114.2,114.2,119.0,119.0,0,0.0,4.4,4.4,-1.0
82,18446901,2025-11-01,2025,12,Indiana Pacers,10,Golden State Warriors,114,109,False,...,3.0,119.0,119.0,116.0,117.4,1,-0.6,-14.4,-13.0,-1.0


In [51]:
# split data into train and validate, including new features

In [52]:
train = model_data[model_data["season"] == 2024]
validation = model_data[model_data["season"] == 2025]

X_train = train[nba_model_features]
y_train = train["home_win"]

X_validation = validation[nba_model_features]
y_validation = validation["home_win"]

In [ ]:
# add in Classifier model

In [70]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

model = DecisionTreeClassifier(
    max_depth=3,
    min_samples_leaf=30,
    random_state=42,
)

model.fit(X_train, y_train)


predictions = model.predict(X_validation)

model_accuracy = accuracy_score(y_validation, predictions)
baseline_accuracy = y_validation.mean()

print(f"Decision tree accuracy: {model_accuracy:.2%}")
print(f"Always-home-win baseline: {baseline_accuracy:.2%}")

print(
    "Training accuracy:",
    f"{accuracy_score(y_train, model.predict(X_train)):.2%}"
)
print(
    "Validation accuracy:",
    f"{accuracy_score(y_validation, model.predict(X_validation)):.2%}"
)
print("Always-home baseline:", f"{y_validation.mean():.2%}")

Decision tree accuracy: 67.50%
Always-home-win baseline: 55.27%
Training accuracy: 67.66%
Validation accuracy: 67.50%
Always-home baseline: 55.27%


In [71]:
# noting above that Decision tree accuracy inched up to 67.5% from 62.51% last week,
# this is a small win, based only on the additional features added so far, next I'll get into the Overfitting/Underfitting exercise
# from Kaggle.com

In [78]:
from sklearn.metrics import mean_absolute_error

def get_mae(max_leaf_nodes, train_X, val_X, train_y, val_y):
    model = DecisionTreeClassifier(max_leaf_nodes=max_leaf_nodes, random_state=0)
    model.fit(train_X, train_y)
    preds_val = model.predict(val_X)
    mae = mean_absolute_error(val_y, preds_val)
    return(mae)

In [81]:
max_nodes = [2, 3, 4, 5, 10, 50]

In [82]:
for node in max_nodes:
    my_mae = get_mae(node, X_train, X_validation, y_train, y_validation)
    print(f"Max leaf nodes: {node}\tMAE: {my_mae:.4f}")

Max leaf nodes: 2	MAE: 0.3113
Max leaf nodes: 3	MAE: 0.3113
Max leaf nodes: 4	MAE: 0.3113
Max leaf nodes: 5	MAE: 0.3250
Max leaf nodes: 10	MAE: 0.3258
Max leaf nodes: 50	MAE: 0.4055
